In [ ]:
import pandas as pd
import numpy as np

folder = '../trictrac_database'

In [ ]:
# avis_clean = pd.read_csv(f'{folder}/avis_clean.csv', header=None, names=["Game id", "User id", "Game name UI", "Username", "Datetime", "Rating", "Comment title", "Comment body"]).drop_duplicates()
# users = pd.read_csv(f'{folder}/users.csv', header=None, names = ["Username", "User id"]).drop_duplicates()
# avis_clean["Datetime"] = pd.to_datetime(avis_clean["Datetime"]) # convert from object to datetime
jeux_clean = pd.read_csv(f'{folder}/jeux_clean.csv', header=None, names = ["Game id", "Game name website", "Game name UI", "Game name year", "Description", "Type", "Extra"])
jeux_clean[jeux_clean["Game name website"] == 'cave-evil']


In [ ]:

# Separate extra info (number of player, age categories, game duration), replace ~ by NaN
jeux_clean[["Number of players", "Age", "Game duration"]] = jeux_clean.pop("Extra").str.split('|',n=3, expand=True, regex=False).replace('~', np.nan)
jeux_clean[jeux_clean["Game id"] == 99]

In [ ]:
# Number of players separation
jeux_clean["Number of players"] = jeux_clean["Number of players"].replace(r"jusqu\'à (\d+)", r"1 à \1", regex=True) # 'jusqu'à num' to '1 à num'
jeux_clean["Number of players"] = jeux_clean["Number of players"].replace(r"à partir de (\d+)", r"\1 à 0", regex=True) # 'à partir de num' to 'num à 0'
jeux_clean["Number of players"] = jeux_clean["Number of players"].replace(r"^\s*(\d+)\s*$", r"\1 à \1", regex=True) # 'num' to 'num à num'

jeux_clean[["Min number of players", "Max number of players"]] = jeux_clean.pop("Number of players").str.extract(r"(\d+)\s*à\s*(\d+)")
# Set types
jeux_clean["Min number of players"] = jeux_clean["Min number of players"].astype('Int16')
jeux_clean["Max number of players"] = jeux_clean["Max number of players"].astype('Int16')
jeux_clean["Max number of players"] = jeux_clean["Max number of players"].replace(0, np.nan,  regex=False)
jeux_clean[jeux_clean["Game id"] == 99]

In [ ]:
jeux_clean[(jeux_clean["Min number of players"] > 10) | (jeux_clean["Max number of players"] > 10)]

In [ ]:
# Age sepration
jeux_clean["Age"] = jeux_clean["Age"].replace(r"(\d+) ans et \+", r"\1 à 99", regex=True) # max age is fixed to 99
jeux_clean[["Age min", "Age max"]] = jeux_clean.pop("Age").str.extract(r"(\d+) à (\d+)")

In [ ]:
# Type conversion
jeux_clean["Age max"] = jeux_clean["Age max"].astype('Int64')
jeux_clean["Age min"] = jeux_clean["Age min"].astype('Int64')
jeux_clean["Game duration"] = jeux_clean["Game duration"].astype('Int64')
jeux_clean.loc[jeux_clean["Age max"] > 99, "Age max"] = 99
jeux_clean[jeux_clean["Game id"] == 99]

In [ ]:
print(jeux_clean.shape)
jeux_clean.drop_duplicates()


In [ ]:
jeux_clean.to_csv('../database_cleaned/jeux_clean.csv')

In [ ]:
# separate number > 10 into first digit and the remainning ones
def digit_sep(n):
    nb_dig = int(np.log10(n))
    p = np.power(10, nb_dig)
    return n // p, n % p

condition = (jeux_clean["Min number of players"] >= 12) & (jeux_clean["Max number of players"].isna())

#for v in jeux_clean.loc[condition, "Min number of players"]:
    #print(v, digit_sep(v))
#jeux_clean_2 = jeux_clean.copy()
jeux_clean.loc[condition, ["Min number of players", "Max number of players"]] = jeux_clean.loc[condition, "Min number of players"].apply(lambda v : digit_sep(v)).to_list()
jeux_clean.to_csv("../database_cleaned/jeux_clean.csv")

In [ ]:
jeux_clean

In [ ]:
jeux_clean.sort_values(by=["Min number of players"], ascending=False)[:30]
jeux_clean[(jeux_clean["Min number of players"] >= 10) & (~jeux_clean["Max number of players"].isna())]

In [ ]:
from collections import Counter
categories = [cat for cats in jeux_clean["Type"][~jeux_clean["Type"].isna()].str.split('|') for cat in cats]
count_cats = Counter(categories)
count_cats

In [ ]:
id_games = jeux_clean[["Game id", "Game name UI"]].groupby(by="Game name UI").count()
id_games[id_games["Game id"] > 1]

jeux_clean[jeux_clean["Game name UI"] == "Waterloo"]

In [ ]:
jeux_clean["Age min"].describe()

In [ ]:
jeux_clean.loc[(~jeux_clean["Min number of players"].isna()) & (jeux_clean["Max number of players"].isna())].shape
for column in jeux_clean.columns:
    print(column, jeux_clean[column].isnull().sum())


In [ ]:
jeux_clean.sort_values(by="Game duration", ascending=False).head(30)

In [ ]:
# Aucune description check
jeux_clean[jeux_clean["Description"].str.contains(r"[a-z]*Aucune[a-z]*", regex=True)].shape[0] / jeux_clean.shape[0]